# Weather-Aware Planner for a Field Technician

This notebook contains code cells for this project which was done as a team project for my Artificial Intelligence course in my final year as a CS student. 


In [4]:
# Uncomment if needed:
# %pip install pandas numpy requests scikit-learn

from __future__ import annotations

import logging
import math
import random
from dataclasses import asdict, dataclass
from datetime import date, datetime, timedelta
from typing import Any

import numpy as np
import pandas as pd
import requests
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')
logger = logging.getLogger('standalone_notebook')
pd.set_option('display.max_columns', 50)

RANDOM_SEED = 42
DEFAULT_TIMEZONE = "Africa/Nairobi"
DEFAULT_WORK_HOURS = (8, 17)
DEFAULT_START_LOCATION = "Nakuru Town"
DEFAULT_SERVICE_TOWNS = ["Nakuru Town", "Naivasha", "Molo", "Gilgil", "Njoro", "Rongai"]
MAX_FORECAST_DAYS = 16
DEFAULT_BEAM_WIDTH = 12
OPEN_METEO_GEOCODE_URL = "https://geocoding-api.open-meteo.com/v1/search"
OPEN_METEO_FORECAST_URL = "https://api.open-meteo.com/v1/forecast"
HOURLY_VARIABLES = ["temperature_2m", "relative_humidity_2m", "precipitation_probability", "precipitation", "cloud_cover", "wind_speed_10m", "wind_gusts_10m"]
MODEL_FEATURES = ["hour", "rain_prob", "precipitation_mm", "cloud_cover", "wind_kph", "gust_kph", "thunder_prob", "temp_c", "humidity", "visibility_km", "is_afternoon", "exposure_hint", "location", "season"]
CAT_COLS = ["location", "season"]
VALID_RISK_LABELS = ["safe", "risky", "unsafe"]
PRIORITY_WEIGHT = {"High": 3, "Medium": 2, "Low": 1}
RISK_PENALTY = {"safe": 0.0, "risky": 1.0, "unsafe": 5.0}
RISK_THRESHOLDS = {
    "unsafe_rain_prob": 0.75, "unsafe_precip_mm": 5.0, "unsafe_gust_kph": 45.0, "unsafe_visibility_km": 4.0, "unsafe_thunder_prob": 0.6,
    "risky_rain_prob": 0.45, "risky_precip_mm": 1.5, "risky_gust_kph": 30.0, "risky_visibility_km": 7.0, "risky_thunder_prob": 0.3,
}
TRAVEL_MINUTES = {
    ("Nakuru Town", "Naivasha"): 95, ("Nakuru Town", "Molo"): 55, ("Nakuru Town", "Gilgil"): 50, ("Nakuru Town", "Njoro"): 35, ("Nakuru Town", "Rongai"): 65,
    ("Naivasha", "Molo"): 120, ("Naivasha", "Gilgil"): 45, ("Naivasha", "Njoro"): 95, ("Naivasha", "Rongai"): 150,
    ("Molo", "Gilgil"): 75, ("Molo", "Njoro"): 45, ("Molo", "Rongai"): 40, ("Gilgil", "Njoro"): 80, ("Gilgil", "Rongai"): 125, ("Njoro", "Rongai"): 80,
}

@dataclass(frozen=True)
class SiteMetadata:
    name: str
    query_name: str
    latitude: float
    longitude: float
    exposure_hint: int
    site_class: str

SITE_METADATA = {
    "Nakuru Town": SiteMetadata("Nakuru Town", "Nakuru, Kenya", -0.3031, 36.0800, 0, "urban"),
    "Naivasha": SiteMetadata("Naivasha", "Naivasha, Kenya", -0.7175, 36.4310, 1, "mixed"),
    "Molo": SiteMetadata("Molo", "Molo, Nakuru County, Kenya", -0.2488, 35.7324, 2, "exposed"),
    "Gilgil": SiteMetadata("Gilgil", "Gilgil, Kenya", -0.4989, 36.3167, 1, "mixed"),
    "Njoro": SiteMetadata("Njoro", "Njoro, Kenya", -0.3341, 35.9428, 1, "mixed"),
    "Rongai": SiteMetadata("Rongai", "Rongai, Kenya", -0.1734, 35.8638, 2, "exposed"),
}
EXPOSURE_HINT_MAP = {name: meta.exposure_hint for name, meta in SITE_METADATA.items()}

@dataclass(frozen=True)
class Task:
    name: str
    location: str
    priority: str
    duration_h: int
    is_outdoor: bool

@dataclass
class ForecastBundle:
    forecast: pd.DataFrame
    source: str
    messages: list[str]
    resolved_locations: dict[str, dict[str, float]]

def derive_season(month: int) -> str:
    return "Wet" if month in {3, 4, 5, 10, 11, 12} else "Dry"

def default_tasks() -> list[Task]:
    return [
        Task("Transformer inspection", "Njoro", "High", 2, True),
        Task("Feeder patrol", "Rongai", "High", 3, True),
        Task("Meter replacement", "Nakuru Town", "Low", 1, False),
        Task("Substation visual check", "Nakuru Town", "High", 1, False),
        Task("Pole repair", "Molo", "Medium", 2, True),
        Task("Customer safety audit", "Naivasha", "Low", 1, False),
        Task("Switchgear inspection", "Gilgil", "Medium", 2, False),
    ]

def derive_thunder_prob(frame: pd.DataFrame) -> pd.Series:
    rain_prob = pd.Series(frame.get("rain_prob", 0.0), index=frame.index).astype(float)
    precipitation = pd.Series(frame.get("precipitation_mm", 0.0), index=frame.index).astype(float)
    cloud_cover = pd.Series(frame.get("cloud_cover", 0.0), index=frame.index).astype(float)
    gust_kph = pd.Series(frame.get("gust_kph", 0.0), index=frame.index).astype(float)
    return (rain_prob * 0.45 + (precipitation / 10.0).clip(lower=0.0, upper=0.35) + (cloud_cover / 100.0) * 0.15 + (gust_kph / 80.0) * 0.10).clip(lower=0.0, upper=1.0)

def derive_visibility_proxy(frame: pd.DataFrame) -> pd.Series:
    rain_prob = pd.Series(frame.get("rain_prob", 0.0), index=frame.index).astype(float)
    precipitation = pd.Series(frame.get("precipitation_mm", 0.0), index=frame.index).astype(float)
    humidity = pd.Series(frame.get("humidity", 50.0), index=frame.index).astype(float)
    cloud_cover = pd.Series(frame.get("cloud_cover", 50.0), index=frame.index).astype(float)
    return (15 - rain_prob * 6 - precipitation * 0.5 - (humidity / 100.0) * 1.8 - (cloud_cover / 100.0) * 1.2).clip(lower=1.0, upper=15.0)

def simulate_weather_rows(n_rows: int = 6000, seed: int = RANDOM_SEED) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    py_rng = random.Random(seed)
    rows = []
    for _ in range(n_rows):
        location = py_rng.choice(DEFAULT_SERVICE_TOWNS)
        hour = py_rng.choice(list(range(6, 20)))
        month = py_rng.choice(list(range(1, 13)))
        season = derive_season(month)
        site = SITE_METADATA[location]
        base_rain = 0.14 if season == "Dry" else 0.48
        rain_prob = np.clip(rng.normal(base_rain + (0.12 if hour >= 13 else 0.02) + (0.08 if site.exposure_hint >= 2 else 0.03 * site.exposure_hint), 0.18), 0, 1)
        precipitation_mm = np.clip(rng.normal((rain_prob ** 2) * 7.5, 1.8), 0, 16)
        cloud_cover = np.clip(rng.normal(42 + rain_prob * 48, 18), 5, 100)
        wind_kph = np.clip(rng.normal(12 + rain_prob * 15 + site.exposure_hint * 3, 7), 0, 70)
        gust_kph = np.clip(wind_kph + rng.normal(8 + rain_prob * 10, 5), wind_kph, 90)
        temp_c = np.clip(rng.normal(24 - rain_prob * 4, 3), 10, 33)
        humidity = np.clip(rng.normal(52 + rain_prob * 35, 12), 25, 99)
        thunder_prob = np.clip(0.10 + rain_prob * 0.45 + (precipitation_mm / 12) * 0.25 + (cloud_cover / 100) * 0.12 + (gust_kph / 80) * 0.08, 0, 1)
        visibility_km = np.clip(15 - rain_prob * 6 - precipitation_mm * 0.55 - (humidity / 100) * 1.8 - (cloud_cover / 100) * 1.2, 1, 15)
        unsafe_score = sum([rain_prob > 0.75, thunder_prob > 0.60, visibility_km < 4.0, gust_kph > 45, precipitation_mm > 5.0])
        risky_score = sum([rain_prob > 0.45, thunder_prob > 0.30, visibility_km < 7.0, gust_kph > 30, precipitation_mm > 1.5])
        risk_label = "unsafe" if unsafe_score >= 2 else ("risky" if risky_score >= 2 else "safe")
        rows.append({"location": location, "hour": hour, "season": season, "rain_prob": float(rain_prob), "precipitation_mm": float(precipitation_mm), "cloud_cover": float(cloud_cover), "wind_kph": float(wind_kph), "gust_kph": float(gust_kph), "thunder_prob": float(thunder_prob), "temp_c": float(temp_c), "humidity": float(humidity), "visibility_km": float(visibility_km), "is_afternoon": int(hour >= 13), "exposure_hint": int(site.exposure_hint), "risk_label": risk_label})
    anchors = pd.DataFrame([
        {"location": "Nakuru Town", "hour": 9, "season": "Dry", "rain_prob": 0.05, "precipitation_mm": 0.0, "cloud_cover": 18.0, "wind_kph": 9.0, "gust_kph": 14.0, "thunder_prob": 0.04, "temp_c": 25.0, "humidity": 45.0, "visibility_km": 13.5, "is_afternoon": 0, "exposure_hint": 0, "risk_label": "safe"},
        {"location": "Gilgil", "hour": 14, "season": "Wet", "rain_prob": 0.58, "precipitation_mm": 2.6, "cloud_cover": 78.0, "wind_kph": 24.0, "gust_kph": 34.0, "thunder_prob": 0.41, "temp_c": 21.0, "humidity": 79.0, "visibility_km": 6.2, "is_afternoon": 1, "exposure_hint": 1, "risk_label": "risky"},
        {"location": "Molo", "hour": 15, "season": "Wet", "rain_prob": 0.92, "precipitation_mm": 9.0, "cloud_cover": 96.0, "wind_kph": 41.0, "gust_kph": 58.0, "thunder_prob": 0.82, "temp_c": 18.0, "humidity": 92.0, "visibility_km": 2.8, "is_afternoon": 1, "exposure_hint": 2, "risk_label": "unsafe"},
    ] * 30)
    return pd.concat([pd.DataFrame(rows), anchors], ignore_index=True)

def train_model(n_rows: int = 6000, seed: int = RANDOM_SEED):
    data = simulate_weather_rows(n_rows=n_rows, seed=seed)
    X = data[MODEL_FEATURES]
    y = data["risk_label"]
    if y.nunique() < 2:
        return None, data["risk_label"].value_counts().to_frame("count"), data
    try:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.22, random_state=seed, stratify=y)
    except ValueError:
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.22, random_state=seed)
    if y_train.nunique() < 2 or y_test.nunique() < 2:
        return None, data["risk_label"].value_counts().to_frame("count"), data
    preprocess = ColumnTransformer([
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median"))]), [c for c in MODEL_FEATURES if c not in CAT_COLS]),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", OneHotEncoder(handle_unknown="ignore"))]), CAT_COLS),
    ])
    forest = Pipeline([("prep", preprocess), ("model", RandomForestClassifier(n_estimators=300, random_state=seed, min_samples_leaf=2, n_jobs=-1))])
    forest.fit(X_train, y_train)
    metrics = pd.DataFrame(classification_report(y_test, forest.predict(X_test), output_dict=True)).transpose().round(3)
    return forest, metrics, data


In [5]:
def prepare_forecast_features(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    if "hour" not in frame.columns and "time" in frame.columns:
        frame["hour"] = pd.to_datetime(frame["time"]).dt.hour
    if "season" not in frame.columns:
        frame["season"] = pd.to_datetime(frame["time"]).dt.month.map(derive_season) if "time" in frame.columns else "Wet"
    if "is_afternoon" not in frame.columns:
        frame["is_afternoon"] = (frame["hour"] >= 13).astype(int)
    if "exposure_hint" not in frame.columns:
        frame["exposure_hint"] = frame["location"].map(EXPOSURE_HINT_MAP).fillna(1).astype(int)
    if "thunder_prob" not in frame.columns:
        frame["thunder_prob"] = derive_thunder_prob(frame)
    if "visibility_km" not in frame.columns:
        frame["visibility_km"] = derive_visibility_proxy(frame)
    for column in MODEL_FEATURES:
        if column not in frame.columns:
            frame[column] = 0.0 if column in {"rain_prob", "thunder_prob", "precipitation_mm", "cloud_cover", "wind_kph", "gust_kph", "temp_c", "humidity", "visibility_km"} else (0 if column in {"is_afternoon", "exposure_hint", "hour"} else ("Wet" if column == "season" else "Unknown"))
    frame["location"] = frame["location"].astype(str)
    frame["season"] = frame["season"].astype(str)
    return frame[MODEL_FEATURES]

def rule_based_risk_labels(forecast_df: pd.DataFrame) -> pd.Series:
    frame = prepare_forecast_features(forecast_df)
    unsafe = ((frame["rain_prob"] >= RISK_THRESHOLDS["unsafe_rain_prob"]).astype(int) + (frame["thunder_prob"] >= RISK_THRESHOLDS["unsafe_thunder_prob"]).astype(int) + (frame["visibility_km"] <= RISK_THRESHOLDS["unsafe_visibility_km"]).astype(int) + (frame["gust_kph"] >= RISK_THRESHOLDS["unsafe_gust_kph"]).astype(int) + (frame["precipitation_mm"] >= RISK_THRESHOLDS["unsafe_precip_mm"]).astype(int))
    risky = ((frame["rain_prob"] >= RISK_THRESHOLDS["risky_rain_prob"]).astype(int) + (frame["thunder_prob"] >= RISK_THRESHOLDS["risky_thunder_prob"]).astype(int) + (frame["visibility_km"] <= RISK_THRESHOLDS["risky_visibility_km"]).astype(int) + (frame["gust_kph"] >= RISK_THRESHOLDS["risky_gust_kph"]).astype(int) + (frame["precipitation_mm"] >= RISK_THRESHOLDS["risky_precip_mm"]).astype(int))
    labels = pd.Series("safe", index=frame.index, dtype="object")
    labels = labels.mask(risky >= 2, "risky")
    labels = labels.mask(unsafe >= 2, "unsafe")
    return labels

def predict_risk(model: Pipeline | None, forecast_df: pd.DataFrame) -> pd.DataFrame:
    prepared = forecast_df.copy()
    features = prepare_forecast_features(prepared)
    predictions = rule_based_risk_labels(prepared) if model is None else pd.Series(model.predict(features), index=prepared.index).astype(str)
    invalid = ~predictions.isin(VALID_RISK_LABELS)
    if invalid.any():
        predictions.loc[invalid] = rule_based_risk_labels(prepared.loc[invalid])
    prepared["pred_risk"] = predictions
    return prepared

class WeatherAPIClient:
    def __init__(self, timeout: int = 20, timezone: str = DEFAULT_TIMEZONE, session: requests.Session | None = None):
        self.timeout = timeout
        self.timezone = timezone
        self.session = session or requests.Session()
    def _candidate_place_names(self, place_name: str) -> list[str]:
        stripped = place_name.replace(", Kenya", "").replace(", Nakuru County", "").strip()
        return list(dict.fromkeys([v for v in [place_name, f"{stripped}, Nakuru County, Kenya", f"{stripped}, Nakuru, Kenya", f"{stripped}, Kenya", stripped] if v]))
    def _select_best_geocode_result(self, results: list[dict[str, Any]]) -> dict[str, Any] | None:
        kenya = [r for r in results if str(r.get("country", "")).lower() == "kenya"]
        nakuru = [r for r in kenya if "nakuru" in str(r.get("admin1", "")).lower() or "nakuru" in str(r.get("admin2", "")).lower() or "nakuru" in str(r.get("admin3", "")).lower()]
        choice = nakuru or kenya or results
        return choice[0] if choice else None
    def geocode_location(self, place_name: str):
        for candidate in self._candidate_place_names(place_name):
            try:
                response = self.session.get(OPEN_METEO_GEOCODE_URL, params={"name": candidate, "count": 5, "language": "en", "format": "json"}, timeout=self.timeout)
                response.raise_for_status()
                choice = self._select_best_geocode_result(response.json().get("results") or [])
            except requests.RequestException:
                choice = None
            if choice:
                return float(choice["latitude"]), float(choice["longitude"]), str(choice.get("name", candidate))
        return None
    def get_hourly_forecast(self, latitude: float, longitude: float, target_date: str) -> pd.DataFrame:
        response = self.session.get(OPEN_METEO_FORECAST_URL, params={"latitude": latitude, "longitude": longitude, "hourly": ",".join(HOURLY_VARIABLES), "start_date": target_date, "end_date": target_date, "timezone": self.timezone}, timeout=self.timeout)
        response.raise_for_status()
        hourly = response.json().get("hourly")
        if not isinstance(hourly, dict) or "time" not in hourly:
            return pd.DataFrame()
        frame = pd.DataFrame(hourly)
        if frame.empty:
            return frame
        for column in {"time", "temperature_2m", "relative_humidity_2m", "precipitation_probability", "precipitation", "cloud_cover", "wind_speed_10m", "wind_gusts_10m"}.difference(frame.columns):
            frame[column] = 0.0
        frame["time"] = pd.to_datetime(frame["time"])
        frame["hour"] = frame["time"].dt.hour
        frame = frame.rename(columns={"temperature_2m": "temp_c", "relative_humidity_2m": "humidity", "precipitation_probability": "rain_prob", "precipitation": "precipitation_mm", "wind_speed_10m": "wind_kph", "wind_gusts_10m": "gust_kph"})
        frame["rain_prob"] = (frame["rain_prob"].astype(float) / 100.0).clip(0.0, 1.0)
        frame["wind_kph"] = frame["wind_kph"].astype(float) * 3.6
        frame["gust_kph"] = frame["gust_kph"].astype(float) * 3.6
        frame["latitude"] = float(latitude)
        frame["longitude"] = float(longitude)
        return frame

def build_scheduler_fallback(target_date: str, towns: list[str] | None = None) -> pd.DataFrame:
    rng = np.random.default_rng(int(target_date.replace("-", "")) % (2**32 - 1))
    rows = []
    for town in towns or DEFAULT_SERVICE_TOWNS:
        site = SITE_METADATA[town]
        for hour in range(DEFAULT_WORK_HOURS[0], DEFAULT_WORK_HOURS[1]):
            rain_prob = np.clip(0.16 + 0.12 * (hour >= 13) + 0.03 * site.exposure_hint + rng.normal(0, 0.08), 0, 1)
            precipitation_mm = np.clip(rng.normal((rain_prob ** 2) * 6, 1.5), 0, 14)
            cloud_cover = np.clip(rng.normal(50 + rain_prob * 35, 15), 10, 100)
            wind_kph = np.clip(rng.normal(11 + site.exposure_hint * 3 + rain_prob * 10, 5), 0, 55)
            gust_kph = np.clip(wind_kph + rng.normal(6 + rain_prob * 8, 4), wind_kph, 75)
            temp_c = np.clip(rng.normal(23 - rain_prob * 3, 2.8), 10, 32)
            humidity = np.clip(rng.normal(58 + rain_prob * 24, 10), 20, 100)
            rows.append({"time": pd.Timestamp(f"{target_date} {hour:02d}:00:00"), "hour": hour, "location": town, "latitude": site.latitude, "longitude": site.longitude, "temp_c": float(temp_c), "humidity": float(humidity), "rain_prob": float(rain_prob), "precipitation_mm": float(precipitation_mm), "cloud_cover": float(cloud_cover), "wind_kph": float(wind_kph), "gust_kph": float(gust_kph)})
    frame = pd.DataFrame(rows)
    frame["thunder_prob"] = derive_thunder_prob(frame)
    frame["visibility_km"] = derive_visibility_proxy(frame)
    frame["season"] = pd.to_datetime(frame["time"]).dt.month.map(derive_season)
    frame["is_afternoon"] = (frame["hour"] >= 13).astype(int)
    frame["exposure_hint"] = frame["location"].map(EXPOSURE_HINT_MAP).astype(int)
    return frame

def get_forecast(date_label: str | None = None, forecast_mode: str = "auto", towns: list[str] | None = None) -> ForecastBundle:
    towns = towns or DEFAULT_SERVICE_TOWNS
    target_date = date.today().isoformat() if date_label is None else date_label
    today = date.today()
    try:
        requested = datetime.strptime(target_date, "%Y-%m-%d").date()
    except ValueError:
        return ForecastBundle(build_scheduler_fallback(today.isoformat(), towns), "fallback", [f"Invalid date {target_date}. Using deterministic fallback forecast."], {})
    if requested < today or requested > today + timedelta(days=MAX_FORECAST_DAYS) or forecast_mode == "fallback":
        msg = "Using deterministic fallback forecast by request." if forecast_mode == "fallback" else f"Requested date {target_date} is outside Open-Meteo forecast range; activating deterministic fallback forecast."
        return ForecastBundle(build_scheduler_fallback(target_date, towns), "fallback", [msg], {})
    client = WeatherAPIClient()
    resolved, messages, rows = {}, [], []
    for town in towns:
        site = SITE_METADATA[town]
        geo = client.geocode_location(site.query_name)
        if geo is None:
            messages.append(f"Geocoding failed for {town}; using hardcoded coordinates.")
            resolved[town] = {"latitude": site.latitude, "longitude": site.longitude}
        else:
            resolved[town] = {"latitude": geo[0], "longitude": geo[1]}
    for town, coords in resolved.items():
        try:
            forecast = client.get_hourly_forecast(coords["latitude"], coords["longitude"], target_date)
        except requests.RequestException as exc:
            messages.append(f"Forecast fetch failed for {town}: {exc}.")
            continue
        if forecast.empty:
            continue
        forecast = forecast[(forecast["hour"] >= DEFAULT_WORK_HOURS[0]) & (forecast["hour"] < DEFAULT_WORK_HOURS[1])].copy()
        if forecast.empty:
            continue
        forecast["location"] = town
        forecast["latitude"] = coords["latitude"]
        forecast["longitude"] = coords["longitude"]
        rows.append(forecast[["time", "hour", "location", "latitude", "longitude", "temp_c", "humidity", "rain_prob", "precipitation_mm", "cloud_cover", "wind_kph", "gust_kph"]])
    if not rows:
        return ForecastBundle(build_scheduler_fallback(target_date, towns), "fallback", messages + ["Live forecast unavailable for all requested towns; using deterministic fallback forecast."], resolved)
    frame = pd.concat(rows, ignore_index=True)
    frame["thunder_prob"] = derive_thunder_prob(frame)
    frame["visibility_km"] = derive_visibility_proxy(frame)
    frame["season"] = pd.to_datetime(frame["time"]).dt.month.map(derive_season)
    frame["is_afternoon"] = (frame["hour"] >= 13).astype(int)
    frame["exposure_hint"] = frame["location"].map(EXPOSURE_HINT_MAP).astype(int)
    return ForecastBundle(frame.sort_values(["location", "time"]).reset_index(drop=True), "live", messages + ["Using live Open-Meteo hourly forecast data."], resolved)

def travel_minutes_between(a: str, b: str) -> int:
    return 0 if a == b else TRAVEL_MINUTES.get((a, b), TRAVEL_MINUTES.get((b, a), 90))

def hourly_risk(forecast_df: pd.DataFrame, location: str, hour: int) -> str:
    row = forecast_df[(forecast_df["location"] == location) & (forecast_df["hour"] == hour)]
    return "risky" if row.empty else (str(row.iloc[0]["pred_risk"]) if str(row.iloc[0]["pred_risk"]) in VALID_RISK_LABELS else "risky")

def schedule_score(order: list[Task], forecast_df: pd.DataFrame, start_hour: int = DEFAULT_WORK_HOURS[0], end_hour: int = DEFAULT_WORK_HOURS[1], start_location: str = DEFAULT_START_LOCATION):
    current_hour, current_loc, completed, postponed = start_hour, start_location, [], []
    total_score, total_travel_min = 0.0, 0
    for task in order:
        travel = travel_minutes_between(current_loc, task.location)
        travel_h = math.ceil(travel / 60)
        if current_hour + travel_h >= end_hour:
            postponed.append(task); continue
        if travel_h > 0:
            total_travel_min += travel; current_hour += travel_h
        if current_hour + task.duration_h > end_hour:
            postponed.append(task); continue
        hours = list(range(current_hour, current_hour + task.duration_h))
        risks = [hourly_risk(forecast_df, task.location, hour) for hour in hours]
        if task.is_outdoor and any(r == "unsafe" for r in risks):
            postponed.append(task); continue
        reward = PRIORITY_WEIGHT[task.priority] * task.duration_h
        risk_cost = sum(RISK_PENALTY[r] for r in risks) if task.is_outdoor else 0.0
        early_bonus = 0.5 if task.priority == "High" and current_hour <= 11 and any(r == "risky" for r in risks) else 0.0
        total_score += reward + early_bonus - risk_cost
        completed.append((task, current_hour, hours, risks, reward, risk_cost, early_bonus))
        current_hour += task.duration_h
        current_loc = task.location
    total_score -= (total_travel_min / 60) * 0.5
    return total_score, {"completed": completed, "postponed": postponed, "travel_minutes": total_travel_min}

def naive_order(tasks: list[Task]) -> list[Task]:
    return sorted(tasks, key=lambda task: (PRIORITY_WEIGHT[task.priority], task.duration_h), reverse=True)

def beam_search_schedule(tasks: list[Task], forecast_df: pd.DataFrame, beam_width: int = DEFAULT_BEAM_WIDTH, start_location: str = DEFAULT_START_LOCATION):
    beams, best = [([], tasks)], None
    for _ in range(len(tasks)):
        expanded = []
        for prefix, remaining in beams:
            for task in remaining:
                new_prefix = prefix + [task]
                new_remaining = [candidate for candidate in remaining if candidate != task]
                score, details = schedule_score(new_prefix, forecast_df, start_location=start_location)
                expanded.append((new_prefix, new_remaining, score, details))
        expanded.sort(key=lambda item: item[2], reverse=True)
        beams = [(p, r) for p, r, _, _ in expanded[:beam_width]]
        for p, r, s, d in expanded:
            if not r and (best is None or s > best[1]):
                best = (p, s, d)
    return best if best is not None else ([], *schedule_score([], forecast_df, start_location=start_location))


In [6]:
model, metrics, training_data = train_model()
if model is None:
    print("Notebook fell back to rule-based risk classification because the synthetic training split was too narrow.")
else:
    print("Model trained in-memory for standalone notebook use.")
display(metrics)

tasks = default_tasks()
forecast_bundle = get_forecast()
forecast_with_risk = predict_risk(model, forecast_bundle.forecast)
baseline_score, _ = schedule_score(naive_order(tasks), forecast_with_risk)
ai_order, ai_score, ai_details = beam_search_schedule(tasks, forecast_with_risk, beam_width=DEFAULT_BEAM_WIDTH)

scheduled_rows = []
for task, start_hour, hours, risks, reward, risk_cost, early_bonus in ai_details["completed"]:
    scheduled_rows.append({**asdict(task), "start_hour": start_hour, "hours": hours, "risk_window": ", ".join(risks), "reward": reward, "risk_cost": risk_cost, "early_bonus": early_bonus})

print(f"Weather source: {forecast_bundle.source}")
print(f"Baseline score: {baseline_score:.2f}")
print(f"Beam-search score: {ai_score:.2f}")
for message in forecast_bundle.messages:
    print(f"- {message}")

display(pd.DataFrame(scheduled_rows))
display(forecast_with_risk[["location", "time", "rain_prob", "precipitation_mm", "gust_kph", "pred_risk"]].head(18))
display(pd.DataFrame([asdict(task) for task in ai_details["postponed"]]))


Model trained in-memory for standalone notebook use.


,precision,recall,f1-score,support
risky,0.988,0.992,0.990,756.000
safe,0.992,0.992,0.992,372.000
unsafe,0.986,0.972,0.979,212.000
accuracy,0.989,0.989,0.989,0.989
macro avg,0.989,0.985,0.987,1340.000
weighted avg,0.989,0.989,0.989,1340.000


Weather source: live
Baseline score: 10.29
Beam-search score: 12.54
- Using live Open-Meteo hourly forecast data.


,name,location,priority,duration_h,is_outdoor,start_hour,hours,risk_window,reward,risk_cost,early_bonus
0,Substation visual check,Nakuru Town,High,1,False,8,[8],safe,3,0.0,0.0
1,Transformer inspection,Njoro,High,2,True,10,"[10, 11]","risky, risky",6,2.0,0.5
2,Feeder patrol,Rongai,High,3,True,14,"[14, 15, 16]","risky, risky, risky",9,3.0,0.0


,location,time,rain_prob,precipitation_mm,gust_kph,pred_risk
0,Gilgil,2026-03-11 08:00:00,0.23,0.3,19.44,safe
1,Gilgil,2026-03-11 09:00:00,0.15,0.1,21.96,safe
2,Gilgil,2026-03-11 10:00:00,0.25,0.0,29.88,safe
3,Gilgil,2026-03-11 11:00:00,0.25,0.0,40.32,safe
4,Gilgil,2026-03-11 12:00:00,0.23,0.0,55.80,risky
5,Gilgil,2026-03-11 13:00:00,0.20,0.0,54.36,safe
6,Gilgil,2026-03-11 14:00:00,0.18,0.1,58.32,safe
7,Gilgil,2026-03-11 15:00:00,0.18,0.1,53.28,risky
8,Gilgil,2026-03-11 16:00:00,0.20,0.1,56.88,risky
9,Molo,2026-03-11 08:00:00,0.03,0.0,84.24,safe


,name,location,priority,duration_h,is_outdoor
0,Meter replacement,Nakuru Town,Low,1,False
1,Pole repair,Molo,Medium,2,True
2,Customer safety audit,Naivasha,Low,1,False
3,Switchgear inspection,Gilgil,Medium,2,False
